# Steam 게임 추천 머신 — Colab 실행 노트북
위에서부터 셀을 **하나씩** 실행하세요 (셀 클릭 → `Shift + Enter`).
결과는 Google Drive의 `MyDrive/steam-recsys/` 폴더에 저장되어, 런타임이 끊겨도 남습니다.

**v3**: 태그가 비어 있던 게임의 태그·설명을 SteamSpy·Steam 스토어에서 보완합니다 (7-1단계).
결과는 `results_v3/`에 저장되고, 이전 결과(`results/`=v1, `results_v2/`)는 비교용으로 그대로 둡니다.

## 1. 런타임 확인 (RAM)
`Total` 메모리가 12GB 이상이면 충분합니다.

In [ ]:
!free -h
!python --version

## 2. Google Drive 연결
팝업이 뜨면 본인 Google 계정을 선택하고 **허용**을 누르세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/steam-recsys'
os.makedirs(WORK, exist_ok=True)
print('작업 폴더:', WORK)

## 3. 코드 가져오기
**방법 A (처음 한 번):** `steam-recsys.zip`을 업로드합니다. 아래 셀을 실행하면 '파일 선택' 버튼이 나타납니다.

**방법 B (GitHub에 올린 뒤):** 방법 A 셀 대신 그 아래 셀에서 주소를 바꿔 실행하세요.

In [ ]:
# 방법 A: zip 업로드
from google.colab import files
up = files.upload()                      # steam-recsys.zip 선택
!rm -rf /content/steam-recsys
!unzip -q -o steam-recsys.zip -d /content/
%cd /content/steam-recsys
!ls

In [ ]:
# 방법 B: GitHub에서 가져오기 (방법 A를 했다면 실행하지 마세요)
# REPO = 'https://github.com/<본인아이디>/<저장소이름>.git'
# !rm -rf /content/steam-recsys && git clone $REPO /content/steam-recsys
# %cd /content/steam-recsys
# !git rev-parse HEAD        # 보고서에 적을 commit hash

## 4. Kaggle API 토큰 등록 (Colab 보안 비밀 사용)
1. https://www.kaggle.com/settings → **API** → **Generate New Token** → 화면에 나온 토큰 문자열을 **복사**
2. Colab 왼쪽 세로 메뉴의 **🔑 열쇠 아이콘(보안 비밀 / Secrets)** 클릭 → **새 보안 비밀 추가**
3. 이름: `KAGGLE_API_TOKEN`, 값: 복사한 토큰 붙여넣기 → **노트북 액세스** 스위치 켜기
4. 아래 셀 실행 (처음 실행 시 접근 허용 팝업이 뜨면 **액세스 권한 부여**)

토큰을 셀 코드에 직접 붙여넣지 마세요. GitHub에 올라가면 유출됩니다.

In [ ]:
!pip install -q --upgrade kaggle
import os
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
print('토큰 등록됨:', bool(os.environ['KAGGLE_API_TOKEN']))
!kaggle --version

## 5. 데이터 다운로드 (몇 분 걸립니다)
데이터는 Colab 로컬 디스크(`data/raw`)에 받습니다. Drive에 받는 것보다 훨씬 빠릅니다.

In [ ]:
%cd /content/steam-recsys
!kaggle datasets download -d antonkozyriev/game-recommendations-on-steam -p data/raw --unzip
!ls -lh data/raw

## 6. 보고서용 데이터 정보 기록 (접근일)
라이선스와 데이터 버전은 Kaggle 데이터셋 페이지 오른쪽의 **License**, **Version** 항목을 직접 보고 아래 셀의 빈칸에 적으세요.

In [ ]:
import json, datetime
info = {'dataset': 'antonkozyriev/game-recommendations-on-steam',
        'license': '',          # Kaggle 페이지의 License 값을 적으세요
        'version': '',          # Kaggle 페이지의 Version 값을 적으세요
        'accessed_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')}
print(json.dumps(info, indent=2, ensure_ascii=False))
json.dump(info, open(f'{WORK}/dataset_info.json', 'w'), indent=2, ensure_ascii=False)

## 7. 패키지 설치
Colab에는 대부분 이미 설치되어 있어 금방 끝납니다.

In [ ]:
!pip install -q -r requirements.txt

## 7-1. 태그·설명 보완 데이터 받기 (v3, 약 23분, 처음 한 번만)
Kaggle 메타데이터에서 태그와 설명이 비어 있던 게임(v2 실험 기준 505개, 대부분 인기 게임)의 정보를
- 태그: **SteamSpy** (초당 1회 제한)
- 설명: **Steam 스토어 공식 API** (5분에 약 200회 제한)

에서 받아 Drive의 `steam-recsys/supplement/supplement_metadata.jsonl`에 저장합니다.

- 대상 목록은 v2 결과의 `results_v2/missing_metadata.csv`를 씁니다 (전처리 기준과 seed가 같아서 v3에서도 같은 게임들입니다).
- 게임 하나를 받을 때마다 바로 저장하므로, **중간에 끊겨도 이 셀을 다시 실행하면 이어서 받습니다.**
- 태그만 빨리 받고 싶으면 명령 끝에 `--no-description`을 붙이세요 (약 9분).
- 다 받은 뒤 이 파일을 GitHub 저장소의 `data/supplement/`에도 올려 두면 다른 사람이 똑같이 재현할 수 있습니다.

In [ ]:
%cd /content/steam-recsys
SUP = f'{WORK}/supplement/supplement_metadata.jsonl'
!python scripts/fetch_missing_tags.py --missing "$WORK/results_v2/missing_metadata.csv" --out "$SUP"
!wc -l "$SUP"

## 8. 빠른 테스트 (약 1–3분)
앞의 300만 행만 써서 코드가 실제 데이터에서 오류 없이 도는지 먼저 확인합니다. **이 결과는 보고서에 쓰지 마세요.**

In [ ]:
!python run_pipeline.py --data-dir data/raw --out-dir /content/test_run \
    --max-rows 3000000 --min-user-pos 5 --min-item-pos 10 --quick

## 9. 본 실험 (전체 데이터, v3)
15–25분 정도 걸립니다. 실행 중에는 브라우저 탭을 닫지 마세요.
로그는 화면과 `run_log_v3.txt` 파일에 동시에 남습니다.

In [ ]:
%cd /content/steam-recsys
OUT = f'{WORK}/results_v3'
SUP = f'{WORK}/supplement/supplement_metadata.jsonl'
!python run_pipeline.py --data-dir data/raw --out-dir "$OUT" --supplement "$SUP" 2>&1 | tee "$WORK/run_log_v3.txt"

**메모리 부족으로 세션이 다운되면** (`Your session crashed after using all available RAM`):
코드와 데이터 파일은 그대로 남아 있으니 **2단계 셀만 다시 실행**한 뒤, 아래 셀의 `#`을 지우고 실행하세요.
`--max-rows`로 앞부분 2,000만 행만 읽습니다. 이 경우 보고서에 사용한 행 수를 반드시 적어야 합니다.

In [ ]:
%cd /content/steam-recsys
OUT = f'{WORK}/results_v3'
SUP = f'{WORK}/supplement/supplement_metadata.jsonl'
# !python run_pipeline.py --data-dir data/raw --out-dir "$OUT" --supplement "$SUP" --max-rows 20000000 2>&1 | tee "$WORK/run_log_v3.txt"

## 10. 결과 확인

In [ ]:
import pandas as pd, json
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 120)
print('전처리 단계별 현황'); display(pd.read_csv(f'{OUT}/preprocessing_report.csv'))
print('메타데이터 보완 결과'); print(json.dumps(json.load(open(f'{OUT}/metadata_fill.json')), indent=2, ensure_ascii=False))
cols = ['precision@10','recall@10','ndcg@10','recall@20','ndcg@20','coverage@20',
        'recall@20_group0','recall@20_group1','recall@20_group2']
m = pd.read_csv(f'{OUT}/metrics_test.csv', index_col=0)
display(m[[c for c in cols if c in m]].round(4))
print(json.dumps(json.load(open(f'{OUT}/chosen_params.json')), indent=2))
print(json.dumps(json.load(open(f'{OUT}/data_stats.json')), indent=2))

In [ ]:
display(pd.read_csv(f'{OUT}/clusters.csv'))
display(pd.read_csv(f'{OUT}/kmeans_k_selection.csv'))
display(pd.read_csv(f'{OUT}/tuning_val.csv')[['component','param','n_rules','rule_user_coverage','ndcg@20']])
display(pd.read_csv(f'{OUT}/apriori_top_rules.csv').head(20))
display(pd.read_csv(f'{OUT}/significance_test.csv'))
print(json.dumps(json.load(open(f'{OUT}/component_analysis.json')), indent=2, ensure_ascii=False))
miss = pd.read_csv(f'{OUT}/missing_metadata.csv')
print('태그/설명이 없는 게임:', len(miss), '| 메타데이터 행 자체가 없음:', int(miss.metadata_row_exists.eq(False).sum()))
display(miss.head(20))

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f'{OUT}/figures/*.png')):
    print(p.split('/')[-1]); display(Image(p))

### 버전 비교 (v1 / v2 / v3)
Drive에 있는 결과 폴더만 골라서 나란히 보여줍니다.

In [ ]:
import os
k = ['ndcg@20', 'recall@20', 'coverage@20', 'recall@20_group2']
tables = {}
for name, folder in [('v1', 'results'), ('v2', 'results_v2'), ('v3', 'results_v3')]:
    p = f'{WORK}/{folder}/metrics_test.csv'
    if os.path.exists(p):
        tables[name] = pd.read_csv(p, index_col=0)[k]
display(pd.concat(tables, axis=1).round(4))

## 11. 결과 다운로드 (선택)
결과는 이미 Drive에 있지만, 압축 파일로 내 컴퓨터에 받고 싶으면 실행하세요.

In [ ]:
!cd "$WORK" && zip -qr /content/results_v3.zip results_v3 run_log_v3.txt supplement dataset_info.json
from google.colab import files
files.download('/content/results_v3.zip')